# Семинар 06. Применяем сортировку

Сортировка нужна не только для того, чтобы вывести числа по возрастанию. Сегодня получим удобный каталог, построим очередь по заданным правилам, найдём близкие измерения и соберём разрозненные периоды занятости в общий календарь.

Примерно **80% занятия — применение сортировки**, задачи 1–4. Оставшиеся **20% — реализация одного алгоритма**, задача 5. В первых четырёх задачах используем `sorted()` и `.sort()`. Переписывать их устройство для каждой задачи не требуется.

## Цели

- переводить правило сортировки в функцию ключа;
- готовить числовые и текстовые ключи, сохранять порядок равных записей;
- использовать сортировку как первый шаг более сложной обработки;
- проверять, что решение сохраняет входные данные и работает на повторах и границах;
- самостоятельно собрать сортировку подсчётом для небольшого диапазона целых чисел.

## Перед началом

Откройте [лекцию](lecture.ipynb). Для задач нужны списки, словари, функции, циклы и `sorted()` с `key`. Новые библиотеки не понадобятся.

Первую задачу начнём вместе: обсудим ожидаемый результат и напишем ключ в коде. Вторую и третью решайте самостоятельно, обсуждая затруднения. Четвёртую разберём совместно у доски или в онлайн-коде: она показывает, как сортировка помогает дальнейшему проходу по данным. Затем соберём собственную сортировку.

В ноутбуке оставлены заготовки без решений. Проверки закомментированы: после реализации снимите комментарии со всего блока проверок нужной функции. Так исходный ноутбук можно запустить целиком, а готовое решение — сразу проверить. Подсказки к двум задачам скрыты под раскрывающимися блоками; сначала попробуйте предложить свой способ.

Во всех задачах вход соответствует условию. Проверки формата и обработка исключений не нужны. Функции возвращают результат; исходные списки, словари и списки правил изменять нельзя. Глубокое копирование записей не требуется. Под «новым списком» понимается отдельный внешний список, в том числе для пустого входа.

### Коротко вспомним: какой объект меняется

До запуска назовём содержимое `values`, `alias` и `ordered`. Что вернёт `.sort()`? Затем выполним ячейку. Если нам нужен только один наибольший элемент, какую функцию лучше взять вместо полной сортировки?

In [ ]:
values = [30, 10, 20]
alias = values
ordered = sorted(values, reverse=True)
returned = values.sort()
print("Исходный список:", values)
print("Второе имя:", alias)
print("Отдельный результат:", ordered)
print("Возврат sort:", returned)

## Задание 1. Каталог книг: данные и ключ

Каталог пришёл из файла. У каждой книги есть уникальный `id`, название `title` и число страниц `pages`, записанное **строкой**. Ведущие нули допустимы. Названия могут отличаться регистром и иметь пробелы по краям.

Реализуйте `order_books(books)`, возвращающую новый список записей:

1. Сначала книги с меньшим **числом** страниц.
2. При равном числе страниц — название по возрастанию после `strip()` и `casefold()`.
3. При равенстве обоих ключей — прежний порядок записей.

Названия и строки с числом страниц в результате должны остаться в исходном виде. До 100 000 книг, число страниц от 0 до 10 000; строка `pages` не длиннее 10 символов, название — 100. При этих ограничениях достаточно O(n log n) времени.

Начнём вместе с порядка только по числу страниц. Затем добавьте название. Для функции ключа сначала используйте `def`; после проверки попробуйте короткую запись через `lambda`. Объясните, почему `id` не входит в ключ.

In [ ]:
books = [
    {"id": "long", "title": "Длинная книга", "pages": "100"},
    {"id": "fruit", "title": "Яблоко", "pages": "20"},
    {"id": "z-first", "title": " азбука ", "pages": "020"},
    {"id": "a-second", "title": "АЗБУКА", "pages": "20"},
    {"id": "short", "title": "Короткая книга", "pages": "3"},
]


def order_books(books):
    # TODO: верните записи в требуемом порядке, не изменяя их поля.
    pass


# before = [book.copy() for book in books]
# result = order_books(books)
# assert [book["id"] for book in result] == [
#     "short", "z-first", "a-second", "fruit", "long"
# ]
# assert result is not books
# assert books == before
# assert result[1]["title"] == " азбука "
# assert result[1]["pages"] == "020"
# empty = []
# empty_result = order_books(empty)
# assert empty_result == [] and empty_result is not empty

После решения измените задачу: нужны только три книги с наименьшим числом страниц с теми же правилами равенства. Как получить их из готового результата? А если нужна только одна книга с наименьшим ключом — обязательно ли сортировать весь каталог?

Проверьте, что в вашей реализации `"100"` сравнивается с `"20"` как число, а `"020"` и `"20"` дают одинаковый числовой ключ. Две «азбуки» должны сохранить исходный порядок, хотя их `id` расположены в обратном алфавитном порядке.

## Задание 2. Очередь: порядок задаётся правилами

Заявки поступают в порядке регистрации. Каждая — словарь с уникальным строковым `id` и полем `status`. Порядок статусов передан отдельно, например `['urgent', 'normal', 'later']`. Алфавит здесь не помогает.

Реализуйте `order_queue(requests, status_order)`. Верните новый список заявок: группы идут в порядке из `status_order`, внутри каждой группы сохраняется порядок поступления. Список статусов не содержит повторов, каждый статус заявки в нём есть. Для пустого списка заявок список статусов тоже может быть пустым.

До 100 000 заявок и до 10 статусов; длина каждого статуса не больше 20 символов. Используйте стабильность встроенной сортировки. Желаемая оценка — O(n log n + k), где `k` — число статусов. Неизменность входных записей и самого `status_order` — часть условия.

Сначала объясните словами, какое значение ключа должно соответствовать каждому статусу. После решения поменяйте порядок статусов и проверьте, что код функции менять не пришлось.

In [ ]:
requests = [
    {"id": "a", "status": "later"},
    {"id": "z", "status": "urgent"},
    {"id": "b", "status": "normal"},
    {"id": "a2", "status": "urgent"},
    {"id": "c", "status": "normal"},
]
status_order = ["urgent", "normal", "later"]


def order_queue(requests, status_order):
    # TODO: используйте переданный порядок статусов и сохраните порядок внутри групп.
    pass


# before = [request.copy() for request in requests]
# order_before = status_order.copy()
# result = order_queue(requests, status_order)
# assert [request["id"] for request in result] == ["z", "a2", "b", "c", "a"]
# assert [request["id"] for request in order_queue(
#     requests, ["later", "normal", "urgent"]
# )] == ["a", "b", "c", "z", "a2"]
# assert result is not requests
# assert requests == before and status_order == order_before
# empty = []
# empty_result = order_queue(empty, [])
# assert empty_result == [] and empty_result is not empty

Должна ли заявка `a2` обогнать `z` внутри срочной группы? Что произойдёт, если добавить `id` в ключ «для определённости»? Учитывает ли программа требования задачи или случайно придумывает дополнительное правило?

## Задание 3. Ближайшие измерения

Датчик выдал целые значения в произвольном порядке. Нужно найти два измерения с наименьшей абсолютной разностью. Перебирать все пары для большого списка слишком долго.

Реализуйте `closest_pair(values)`. Она возвращает кортеж `(a, b)`, где `a <= b` и оба значения взяты из **разных позиций** входа. При нескольких парах с одинаковой минимальной разностью выберите меньшую пару при обычном сравнении кортежей: сначала меньший `a`, затем меньший `b`. Если элементов меньше двух, верните `None`.

Повторяющиеся значения допустимы: два отдельных измерения `4` и `4` образуют пару с разностью 0. До 200 000 чисел от −10⁹ до 10⁹. Требуется O(n log n) времени или лучше. Исходный список не меняйте; `sorted()` разрешён.

Предложите способ решения. Что станет проще, если сначала упорядочить измерения?

<details>
<summary>Подсказка после самостоятельной попытки</summary>

В упорядоченном списке посмотрите на соседей. Если между двумя выбранными значениями стоит третье, может ли оно находиться дальше от обоих, чем они друг от друга? После этого подумайте, как сохранить лучшую найденную пару за один проход.

</details>

In [ ]:
def closest_pair(values):
    # TODO: найдите ближайшую пару с указанным правилом равенства.
    pass


# values = [20, 3, 14, 9]
# before = values.copy()
# assert closest_pair(values) == (9, 14)
# assert values == before
# assert closest_pair([]) is None
# assert closest_pair([7]) is None
# assert closest_pair([20, 1]) == (1, 20)
# assert closest_pair([9, 4, 4, 12]) == (4, 4)
# assert closest_pair([9, 7, 3, 1]) == (1, 3)
# assert closest_pair([-1, -8, 4, -3]) == (-3, -1)
# assert closest_pair([5, 5, 2, 2]) == (2, 2)

После решения объясните, какую работу выполнила сортировка и какая работа осталась вашему циклу. Сколько пар проверяет полный перебор, а сколько — ваше решение после упорядочивания? Отдельно проверьте случай нескольких одинаково хороших пар.

## Задание 4. Разберём вместе: общий календарь занятости

Для одной комнаты поступили периоды занятости из разных источников. Каждый период — кортеж `(start, end)` с временем в минутах от начала дня: `0 <= start < end <= 1440`. Начало включается в период, конец — нет. Записи могут пересекаться, повторяться и находиться в любом порядке.

Реализуем `merge_busy_periods(periods)`: новый список кортежей, описывающий ту же занятость. Пересекающиеся **и соприкасающиеся** периоды нужно объединить. Например, `(10, 20)` и `(20, 30)` дают `(10, 30)`. Ответ идёт по возрастанию начала; между соседними периодами ответа должен оставаться свободный промежуток. Исходный список не меняем.

До 200 000 периодов; достаточно O(n log n) времени. Встроенная сортировка разрешена. Совместно разберём вход `[(40, 60), (10, 30), (25, 45)]`, затем напишем код. Почему нельзя обрабатывать записи просто в порядке поступления?

<details>
<summary>Вопросы для совместного разбора</summary>

По какому полю сначала упорядочить периоды? Как понять, что очередной период продолжает уже собранный? Что делать, если он целиком лежит внутри? Сравнивать его с предыдущей исходной записью или с последним **объединённым** периодом?

</details>

In [ ]:
def merge_busy_periods(periods):
    # TODO: верните объединённые периоды в порядке времени.
    pass


# periods = [(40, 60), (10, 30), (25, 45)]
# before = periods.copy()
# result = merge_busy_periods(periods)
# assert result == [(10, 60)]
# assert result is not periods
# assert periods == before
# assert all(isinstance(period, tuple) for period in result)
# assert merge_busy_periods([(30, 40), (10, 20)]) == [(10, 20), (30, 40)]
# assert merge_busy_periods([(10, 20), (20, 30)]) == [(10, 30)]
# assert merge_busy_periods([(10, 100), (20, 30), (40, 50)]) == [(10, 100)]
# assert merge_busy_periods([(10, 20), (10, 20)]) == [(10, 20)]
# assert merge_busy_periods([(0, 1440)]) == [(0, 1440)]
# empty = []
# empty_result = merge_busy_periods(empty)
# assert empty_result == [] and empty_result is not empty

Проверим особенно внимательно вложенные периоды: если комната занята с 10 до 100, запись `(20, 30)` не должна сократить занятость до 30. Объясним, почему после сортировки достаточно одного прохода и почему ни один период не теряется.

## Задание 5. Своя сортировка: подсчёт

На этот блок оставляем примерно пятую часть занятия. Вспомним идею из лекции у доски или на нескольких значениях в коде, затем самостоятельно реализуйте `counting_sort(values, max_value)`.

Вход содержит целые числа от `0` до `max_value` включительно. Верните **новый** список по возрастанию. Сохраните все повторы и исходный список. До 200 000 элементов, `0 <= max_value <= 100_000`. Вход корректен: дополнительных проверок и `raise` не требуется.

В реализации нельзя использовать `sorted()` и `.sort()`. В проверках они разрешены как эталон. Требования: O(n + k) времени и O(k) дополнительной памяти сверх результата, где `k = max_value + 1`.

Что хранится по индексу `3` в массиве счётчиков? Для какого входа массив счётчиков станет слишком большим, даже если самих чисел мало?

In [ ]:
def counting_sort(values, max_value):
    # TODO: посчитайте значения и восстановите упорядоченный список.
    pass


# cases = [[], [0], [3, 0, 3, 1], [1, 2, 3], [3, 2, 1], [2, 2, 2]]
# for values in cases:
#     before = values.copy()
#     result = counting_sort(values, 3)
#     assert result == sorted(values)
#     assert result is not values
#     assert values == before
# assert counting_sort([0, 0], 0) == [0, 0]
# assert counting_sort([100_000, 0, 100_000], 100_000) == [0, 100_000, 100_000]

## Итоги

В каталоге мы подготовили ключ, не изменив сами записи. В очереди задали внешний порядок групп и сохранили порядок поступления. В задаче о ближайших измерениях сортировка помогла сократить число рассматриваемых пар. В календаре она позволила объединить периоды одним проходом. Только в последнем задании мы реализовали сам алгоритм сортировки.

Для каждого решения объясните: зачем здесь порядок, по чему сортируем и что делаем с данными после сортировки? Затем проверьте повторы, пустой вход и неизменность исходных данных.

## Домашняя работа

[Основные задачи](tasks.md) — три Easy и две Medium. Для желающих есть отдельная необязательная задача на **radix sort**: стабильные проходы по разрядам неотрицательных целых чисел. Идея и необходимые правила описаны в условии; на семинаре реализовывать ещё один алгоритм не нужно.

> В материалах курса приведены **предполагаемые списки домашних задач**. Домашние задания нужно решать и сдавать в **Яндекс.Контесте**. Актуальный список задач, условия и требования к оформлению находятся в Контесте. **При расхождениях информация в Контесте имеет приоритет.** Ссылки на контесты публикуются на **вики курса**.